# 03 表格与文字断言审计（Tier A）

本 Notebook 不训练任何模型，只做两件事：

1. 把论文 Table 3-7 的数字逐格转录到 `src/ebba/claims/tables.py`，然后用程序检查
   表格内部的算术、下界、阈值与正文声称的“最优/更低”。
2. 静态解析官方 Notebook，对照论文的实验协议。

> 转录原则：数字只从 PDF 抄一次；若结论与论文不符，改的是**判定**，不是表格。
> 所有行数、取值范围由 `tests/test_claims.py` 校验。


In [ ]:
import json
import os
import sys
from pathlib import Path

# 定位仓库根目录：容器内是 /app，本地打开时可能是 notebooks/ 或其子目录。
ROOT = Path(os.environ.get("EBBA_ROOT", Path.cwd())).resolve()
if not (ROOT / "src" / "ebba").exists():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "src" / "ebba").exists():
            ROOT = cand
            break
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("EBBA_ROOT", str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

from ebba import config, nbutil
from ebba.claims import load_registry, run, run_many

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("project root:", ROOT)
print("artifacts   :", config.ARTIFACTS_DIR)
print("reports     :", config.REPORTS_DIR)
print("fast mode   :", config.FAST)


In [ ]:
from ebba.claims import tables

print("Table 3 rows:", len(tables.table3_rows()), "| datasets:",
      sorted({r["dataset"] for r in tables.table3_rows()}))
print("Table 4 rows:", len(tables.table4_rows()), "| methods:", tables.TABLE4_METHODS)
print("Table 5/7 PDR:", tables.TABLE5, "| identical:", tables.TABLE5 == tables.TABLE7)
print("Table 6:", json.dumps(tables.TABLE6, ensure_ascii=False))


## A16：Table 3 是否“全面最优”？

正文称 EBBA+ 在 PDR 与 F1 上优于所有对比方法。逐行比较 15 行 × 2 个指标，
只要存在一个严格劣于最好基线的格子，就记一个反例（平局不算）。


In [ ]:
r = nbutil.show("A16", max_measured_chars=2500)


## A11：BA 下界是否对所有行成立？

Sec. 5.2 给出经验下界

$$\mathrm{BA}\ \ge\ \frac{K-1}{K}\,\mathrm{ba},$$

其中 $K$ 是类别数，$\mathrm{ba}$ 是防御前的良性精度。它不是定理，而是“只改被判为目标类的图”这一操作的
最坏情况估计：每个类别最多损失 $1/K$。下面用 Table 4 的每一行检查（ImageNet 取 $K=100$）。


In [ ]:
r = nbutil.show("A11", max_measured_chars=2200)


## A12：所有攻击的 ASR 是否都训练到 98.8% 以上？

正文用“No Defense”列的 ASR 说明攻击足够强。逐行检查 Table 4 的 No Defense 列即可。


In [ ]:
r = nbutil.show("A12")


## A15：Table 4 的文字结论

正文有两句话需要分开核对：

1. “EBBA+ 的 BA 最优” —— 与防御方法比较（不含 No Defense，因为无防御不是方法）；
2. “EBBA+ 把 ASR 降得更低” —— 与所有方法比较（含 No Defense）。

程序分别统计“BA 严格最好”和“ASR 严格最低”的行数。平局不计为最好/最低。


In [ ]:
r = nbutil.show("A15", max_measured_chars=2500)


## A13：Table 6 的 GTSRB 测试集大小

论文写 12603，GTSRB 官方测试集是 12630。两者只差数字顺序，属于转录错误。
这类错误本身不影响实验，但会让人怀疑其他表格数字的可靠性。


In [ ]:
r = nbutil.show("A13")


## A14：Table 5 与 Table 7 的聚类差异

论文用 Table 5（正文）和 Table 7（附录）说明四种聚类方法影响很小。
两张表**逐格相同**，所以它们不是两条独立证据。这里只检查极差是否 ≤ 0.05，
并记录“同一组数字被印了两次”这一事实。


In [ ]:
r = nbutil.show("A14")


## A18：官方 Notebook 是否实现了论文协议？

官方代码是 PaddlePaddle Notebook（`reference/EBBA.ipynb`）。本项目**不执行**它，
只解析 JSON 并检查 8 项协议：图像池是否任务无关、是否按伪标签均衡、样本量、损坏数量、
除法损坏是否从 1 开始、TE 截断是否一致、是否有聚类步骤、是否报告 BA/ASR。


In [ ]:
r = nbutil.show("A18", max_measured_chars=3200)
for check in r.measured["checks"]:
    mark = "PASS" if check["passes"] else "FAIL"
    print(f'{mark}  {check["id"]:<20} official: {check["official"]}')


## A19：自适应攻击下“检测率超过 99%”

附录 G.4 列出 13 组软标签参数，并明确写道 0.985 时 EBBA 失效。按参数组计算，
通过率至多 12/13 = 92.3%。若“检测率”指逐样本比例，则文中没有给出可复算的数值，
只能记 PARTIAL。


In [ ]:
r = nbutil.show("A19")


## 本 Notebook 的小结

- Table 3 的“PDR/F1 全面最优”和 Table 4 的“BA 最优、ASR 更低”都能找到反例；
- BA 下界不是定理，Table 4 内部就有两行违反；
- “ASR 都超过 98.8%”有两个反例；
- GTSRB 测试集大小印错；
- 官方 Notebook 与论文协议有 7 项不符，只能作为公式参考；
- 自适应攻击的“>99%”按参数组只有 92.3%。

这些结论的含义是：**论文的表述与表格存在可核对的不一致**，需要在引用时注明，而不是说方法无效。
